# Hospital Cleanliness Synthetic Augmentation (FLUX.1 Fill Dev) 🏥

This notebook creates realistic synthetic "dirty" hospital images from "clean" hospital images.
It is optimized to improve a downstream MobileNetV2 cleanliness classifier by generalizing the dataset while keeping the base hospital layout and lighting unchanged.

**Features:**
- 🤖 **AI Inpainting Mode (Default)**: Uses `black-forest-labs/FLUX.1-Fill-dev` to photorealistically hallucinate complex dirt and objects inside the floor mask.
- 🖼️ **Classical CV Overlay Mode (Fast)**: Uses alpha blending, shadow generation, brightness matching, and perspective transforms to place transparent PNG assets realistically on the floor.

*Author: Principal Computer Vision Engineer*


In [ ]:
#@title 1. Install dependencies
!pip install opencv-python numpy Pillow albumentations matplotlib pandas tqdm diffusers transformers accelerate sentencepiece protobuf huggingface_hub bitsandbytes


In [ ]:
#@title 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
#@title 3. Imports
import os
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd
from tqdm.auto import tqdm
import random
import time
from datetime import datetime
import glob
import math

# For optional AI Inpainting Mode
import torch
from diffusers import FluxFillPipeline


In [ ]:
#@title 4. Config
# Configuration Parameters

# Retrieve token from Colab Secrets safely to avoid exposing it on GitHub
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except ImportError:
    import os
    HF_TOKEN = os.getenv("HF_TOKEN")

INPUT_FOLDER = '/content/drive/MyDrive/cleanvision_dataset/clean'  #@param {type:"string"}
OUTPUT_FOLDER = '/content/drive/MyDrive/cleanvision_dataset/generated_dirty'  #@param {type:"string"}
ASSETS_FOLDER = '/content/drive/MyDrive/cleanvision_dataset/assets'  #@param {type:"string"}

# Generation Mode ('classical' or 'ai_inpainting')
MODE = 'ai_inpainting'  #@param ["classical", "ai_inpainting"]

# Execution Mode
EXECUTION = 'preview' #@param ["preview", "batch"]

# Parameters
VARIANTS_PER_IMAGE = 1  #@param {type:"integer"}
MAX_OBJECTS = 3  #@param {type:"integer"}
DIFFICULTY = 'medium'  #@param ["easy", "medium", "hard", "mixed"]

# Output Settings
OUTPUT_SIZE = 224  #@param {type:"integer"}
JPEG_QUALITY = 95  #@param {type:"slider", min:10, max:100, step:1}
SAVE_METADATA = True  #@param {type:"boolean"}
SEED = 42  #@param {type:"integer"}

# Set random seeds for reproducibility
random.seed(SEED)
np.random.seed(SEED)

os.makedirs(OUTPUT_FOLDER, exist_ok=True)


In [ ]:
#@title 5. Utilities
def plot_images(images, titles=None, figsize=(20, 10)):
    """Helper to display multiple images."""
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=figsize)
    if n == 1:
        axes = [axes]
    for i in range(n):
        axes[i].imshow(cv2.cvtColor(images[i], cv2.COLOR_BGR2RGB))
        if titles:
            axes[i].set_title(titles[i])
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()


In [ ]:
#@title 6. Load dataset
print("Scanning input folder...")
formats = ['*.jpg', '*.jpeg', '*.png', '*.webp', '*.bmp', '*.JPG', '*.JPEG', '*.PNG', '*.WEBP', '*.BMP']
clean_images = []
for fmt in formats:
    clean_images.extend(glob.glob(os.path.join(INPUT_FOLDER, fmt)))
    
print(f"Found {len(clean_images)} clean images.")

if len(clean_images) == 0:
    print("WARNING: No images found. Please verify INPUT_FOLDER path.")


In [ ]:
#@title 7. Load PNG assets
def load_assets(assets_path):
    """Loads transparent PNG objects categorized by difficulty."""
    assets = {'easy': [], 'medium': [], 'hard': []}
    
    if MODE != 'classical':
        return assets # Skip loading if using AI mode
        
    if not os.path.exists(assets_path):
        print("WARNING: Assets folder not found. Classical overlay requires transparent PNGs.")
        return assets
        
    for diff in assets.keys():
        diff_path = os.path.join(assets_path, diff)
        if os.path.exists(diff_path):
            files = glob.glob(os.path.join(diff_path, '*.png'))
            for f in files:
                img = cv2.imread(f, cv2.IMREAD_UNCHANGED)
                if img is not None and img.shape[2] == 4:
                    assets[diff].append({'path': f, 'image': img})
    
    return assets

assets = load_assets(ASSETS_FOLDER)


In [ ]:
#@title 8. Floor region detection & Advanced Masking
def get_floor_mask(image):
    """
    Detects the floor region in the image.
    Uses a fast heuristic: bottom 60% of the image is generally floor in hospital corridors.
    Returns a binary mask.
    """
    h, w = image.shape[:2]
    mask = np.zeros((h, w), dtype=np.uint8)
    
    # Bottom 60% as safe floor region, with a slight trapezoidal shape for perspective
    pts = np.array([
        [int(w * 0.1), int(h * 0.4)],
        [int(w * 0.9), int(h * 0.4)],
        [w, h],
        [0, h]
    ], np.int32)
    
    cv2.fillPoly(mask, [pts], 255)
    return mask

def get_random_floor_point(mask, margin=50):
    """Returns a random (x, y) coordinate inside the floor mask."""
    ys, xs = np.where(mask > 0)
    if len(ys) == 0:
        return None
    
    # Avoid edges
    valid = (ys > margin) & (ys < mask.shape[0] - margin) & (xs > margin) & (xs < mask.shape[1] - margin)
    valid_ys = ys[valid]
    valid_xs = xs[valid]
    
    if len(valid_ys) == 0:
        idx = random.randint(0, len(ys)-1)
        return xs[idx], ys[idx]
        
    idx = random.randint(0, len(valid_ys)-1)
    return valid_xs[idx], valid_ys[idx]

def generate_advanced_mask(image_shape, floor_mask):
    """
    Generates a realistic, randomized mask (blob, ellipse, torn paper) safely within the floor boundary.
    Occupies roughly 3-12% of the image area.
    """
    h, w = image_shape[:2]
    total_area = h * w
    target_area = random.uniform(0.03, 0.12) * total_area
    
    # Erode floor mask heavily to ensure objects don't touch walls/furniture
    kernel = np.ones((35,35), np.uint8)
    safe_zone = cv2.erode(floor_mask, kernel, iterations=2)
    
    inpaint_mask = np.zeros((h, w), dtype=np.uint8)
    pt = get_random_floor_point(safe_zone, margin=0)
    
    if not pt:
        # Fallback if erosion kills the mask
        pt = (w//2, int(h*0.75))
        
    cx, cy = pt
    shape_type = random.choice(['ellipse', 'blob', 'polygon'])
    
    # Estimate radius based on target area (A = pi * r^2)
    approx_radius = int(math.sqrt(target_area / math.pi))
    
    if shape_type == 'ellipse':
        axes = (int(approx_radius * random.uniform(0.8, 1.5)), int(approx_radius * random.uniform(0.4, 0.8)))
        angle = random.randint(0, 180)
        cv2.ellipse(inpaint_mask, (cx, cy), axes, angle, 0, 360, 255, -1)
        
    elif shape_type == 'blob':
        # Create an irregular blob using multiple overlapping circles
        for _ in range(random.randint(3, 7)):
            offset_x = random.randint(-approx_radius//2, approx_radius//2)
            offset_y = random.randint(-approx_radius//2, approx_radius//2)
            rad = random.randint(approx_radius//3, approx_radius)
            cv2.circle(inpaint_mask, (cx + offset_x, cy + offset_y), rad, 255, -1)
            
    else: # polygon / torn paper
        num_points = random.randint(5, 10)
        points = []
        for i in range(num_points):
            angle = (i * 2 * math.pi) / num_points
            rad = approx_radius * random.uniform(0.6, 1.4)
            px = int(cx + rad * math.cos(angle))
            py = int(cy + rad * math.sin(angle))
            points.append([px, py])
        pts = np.array(points, np.int32)
        pts = pts.reshape((-1, 1, 2))
        cv2.fillPoly(inpaint_mask, [pts], 255)
        
    # Final check: AND with the safe zone to strictly enforce boundaries
    inpaint_mask = cv2.bitwise_and(inpaint_mask, safe_zone)
    return inpaint_mask


In [ ]:
#@title 9. Classical CV Overlay Pipeline (Skipped in AI Mode)
if MODE == 'classical':
    def generate_soft_shadow(alpha_channel, offset=(10, 10), blur=15, intensity=0.5):
        h, w = alpha_channel.shape
        shadow = np.zeros((h + abs(offset[1]*2) + blur*2, w + abs(offset[0]*2) + blur*2), dtype=np.uint8)
        cy, cx = shadow.shape[0]//2 - h//2, shadow.shape[1]//2 - w//2
        shadow[cy:cy+h, cx:cx+w] = alpha_channel
        M = np.float32([[1, 0, offset[0]], [0, 1, offset[1]]])
        shadow = cv2.warpAffine(shadow, M, (shadow.shape[1], shadow.shape[0]))
        if blur > 0: shadow = cv2.GaussianBlur(shadow, (blur*2+1, blur*2+1), 0)
        shadow = (shadow * intensity).astype(np.uint8)
        return shadow, cx, cy

    def apply_perspective_distortion(asset, intensity=0.2):
        h, w = asset.shape[:2]
        src_pts = np.float32([[0, 0], [w, 0], [w, h], [0, h]])
        dw, dh = w * intensity, h * intensity
        dst_pts = np.float32([
            [random.uniform(0, dw), random.uniform(0, dh)],
            [w - random.uniform(0, dw), random.uniform(0, dh)],
            [w + random.uniform(-dw/2, dw/2), h],
            [-random.uniform(-dw/2, dw/2), h]
        ])
        M = cv2.getPerspectiveTransform(src_pts, dst_pts)
        out_w, out_h = int(w*1.5), int(h*1.5)
        warped = cv2.warpPerspective(asset, M, (out_w, out_h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=(0,0,0,0))
        coords = cv2.findNonZero(warped[:, :, 3])
        if coords is not None:
            x, y, w2, h2 = cv2.boundingRect(coords)
            return warped[y:y+h2, x:x+w2]
        return asset

    def match_brightness(bg_img, fg_asset, cx, cy):
        h, w = fg_asset.shape[:2]
        bh, bw = bg_img.shape[:2]
        y1, y2 = max(0, cy - h//2), min(bh, cy + h//2)
        x1, x2 = max(0, cx - w//2), min(bw, cx + w//2)
        bg_roi = bg_img[y1:y2, x1:x2]
        if bg_roi.size == 0: return fg_asset
        bg_hsv = cv2.cvtColor(bg_roi, cv2.COLOR_BGR2HSV)
        bg_v = np.mean(bg_hsv[:,:,2])
        fg_bgr = fg_asset[:,:,:3]
        fg_hsv = cv2.cvtColor(fg_bgr, cv2.COLOR_HSV2BGR)
        fg_v = np.mean(fg_hsv[:,:,2][fg_asset[:,:,3] > 0]) 
        if fg_v < 10: fg_v = 10
        ratio = np.clip(bg_v / fg_v, 0.4, 1.5) 
        fg_hsv[:,:,2] = np.clip(fg_hsv[:,:,2] * ratio, 0, 255).astype(np.uint8)
        fg_matched = cv2.cvtColor(fg_hsv, cv2.COLOR_HSV2BGR)
        result = fg_asset.copy()
        result[:,:,:3] = fg_matched
        return result

    def overlay_transparent(background, overlay, x, y):
        bg_h, bg_w, bg_channels = background.shape
        fg_h, fg_w, fg_channels = overlay.shape
        if x >= bg_w or y >= bg_h: return background, False
        if x + fg_w <= 0 or y + fg_h <= 0: return background, False
        src_x, src_y = max(0, -x), max(0, -y)
        src_w, src_h = min(fg_w, bg_w - x) - src_x, min(fg_h, bg_h - y) - src_y
        dst_x, dst_y = max(0, x), max(0, y)
        fg_roi = overlay[src_y:src_y+src_h, src_x:src_x+src_w]
        bg_roi = background[dst_y:dst_y+src_h, dst_x:dst_x+src_w]
        alpha = fg_roi[:, :, 3] / 255.0
        fg_rgb = fg_roi[:, :, :3]
        alpha_3d = np.dstack((alpha, alpha, alpha))
        blended = (alpha_3d * fg_rgb) + ((1 - alpha_3d) * bg_roi)
        result = background.copy()
        result[dst_y:dst_y+src_h, dst_x:dst_x+src_w] = blended
        return result, True

    def augment_classical(image_path, num_objects=2, difficulty='mixed'):
        img = cv2.imread(image_path)
        if img is None: return None, []
        floor_mask = get_floor_mask(img)
        result_img = img.copy()
        objects_used = []
        
        for _ in range(num_objects):
            diff = random.choice(['easy', 'medium', 'hard']) if difficulty == 'mixed' else difficulty
            if diff not in assets or len(assets[diff]) == 0: continue
            asset_info = random.choice(assets[diff])
            asset = asset_info['image'].copy()
            
            scale = random.uniform(0.1, 0.4) if diff == 'hard' else random.uniform(0.3, 1.0)
            ah, aw = asset.shape[:2]
            asset = cv2.resize(asset, (int(aw * scale), int(ah * scale)))
            asset = apply_perspective_distortion(asset, intensity=0.3)
            
            pt = get_random_floor_point(floor_mask)
            if not pt: continue
            cx, cy = pt
            
            asset = match_brightness(result_img, asset, cx, cy)
            shadow, scx, scy = generate_soft_shadow(asset[:,:,3], offset=(5, 10), blur=5, intensity=0.6)
            
            fg_x, fg_y = cx - asset.shape[1]//2, cy - asset.shape[0]//2
            sh_x, sh_y = cx - shadow.shape[1]//2 + scx, cy - shadow.shape[0]//2 + scy
            
            shadow_rgba = cv2.cvtColor(shadow, cv2.COLOR_GRAY2BGRA)
            shadow_rgba[:,:,3], shadow_rgba[:,:,:3] = shadow, 0
            
            result_img, success_sh = overlay_transparent(result_img, shadow_rgba, sh_x, sh_y)
            result_img, success_fg = overlay_transparent(result_img, asset, fg_x, fg_y)
            if success_fg: objects_used.append(os.path.basename(asset_info['path']))
                
        return result_img, objects_used
else:
    def augment_classical(image_path, num_objects=2, difficulty='mixed'):
        return None, []


In [ ]:
#@title 14. AI Inpainting Setup (FLUX.1)
ai_pipeline = None

# Expanded to 25 diverse and realistic prompts
PROMPTS = [
    "A single used tissue naturally lying on a clean hospital tile floor.",
    "A disposable surgical mask lying on the hospital floor.",
    "Small muddy shoe footprints leading across the hospital floor.",
    "A transparent water spill reflecting ceiling lights.",
    "A medicine blister wrapper near the wall.",
    "A few tiny paper scraps near the edge of the corridor.",
    "Small dust accumulation in a floor corner.",
    "Minor dirt marks from heavy hospital foot traffic.",
    "A used gauze pad lying naturally on the hospital floor.",
    "Small scattered debris near the floor edge.",
    "Dry brown leaf debris on a hospital tiled hallway floor.",
    "Faint dark scuff marks from shoes on hospital vinyl flooring.",
    "A dropped cotton ball on a polished clinic floor.",
    "A small puddle of yellow antiseptic fluid on the floor.",
    "A single plastic syringe cap dropped near the floor baseboard.",
    "A discarded adhesive bandage strip on the hospital floor.",
    "A patch of grey dust bunny lint in the corner of a corridor.",
    "A small dark coffee stain on the hospital tiles.",
    "A couple of hair strands stuck to a wet spot on the floor.",
    "A dropped wooden tongue depressor on a hospital tile.",
    "Faint wet footprint outlines drying on a shiny floor.",
    "Small crinkled piece of medical tape on the ground.",
    "A dropped blue nitrile glove lying flat on the corridor floor.",
    "A clean sterile gauze wrapper lying empty on the tile floor.",
    "A slight smudge of dark grease near the door frame on the floor."
]

NEGATIVE_PROMPT = "people, furniture, beds, walls changing, doors changing, lighting changes, ceiling changes, extra rooms, duplicated objects, text, watermark, painting, cartoon, CGI, blur, out of focus, unrealistic"

def load_ai_pipeline():
    global ai_pipeline
    if ai_pipeline is None:
        print("Loading black-forest-labs/FLUX.1-Fill-dev Model (optimized for 16GB T4 GPU)...")
        from transformers import BitsAndBytesConfig
        from diffusers import FluxTransformer2DModel
        import torch
        
        # Configure 4-bit quantization for the massive 12B parameter Transformer
        # This reduces its memory size from 24GB to ~6.5GB, allowing it to fit on T4 VRAM
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16
        )
        
        print("  - Loading 4-bit quantized transformer...")
        transformer = FluxTransformer2DModel.from_pretrained(
            "black-forest-labs/FLUX.1-Fill-dev",
            subfolder="transformer",
            quantization_config=quant_config,
            torch_dtype=torch.bfloat16,
            token=HF_TOKEN
        )
        
        print("  - Loading FLUX Fill Pipeline...")
        ai_pipeline = FluxFillPipeline.from_pretrained(
            "black-forest-labs/FLUX.1-Fill-dev", 
            transformer=transformer,
            torch_dtype=torch.bfloat16,
            token=HF_TOKEN
        )
        
        # Enable aggressive CPU offload and slicing to stay well within VRAM limits
        print("  - Applying memory optimizations...")
        ai_pipeline.enable_model_cpu_offload() 
        try:
            ai_pipeline.enable_vae_slicing()
            ai_pipeline.enable_vae_tiling()
            ai_pipeline.enable_attention_slicing()
        except Exception as e:
            print(f"Warning: Model optimization initialization details: {e}")

def augment_ai_inpainting(image_path, seed=None):
    """
    Uses AI inpainting to hallucinate dirt on the floor using FLUX.1 Fill Dev.
    """
    import gc
    import torch
    
    # Aggressive garbage collection before generation to prevent VRAM fragmentation
    gc.collect()
    torch.cuda.empty_cache()

    prompt = random.choice(PROMPTS)
    load_ai_pipeline()
    
    # Read original image
    img = Image.open(image_path).convert("RGB")
    orig_w, orig_h = img.size
    
    # Internal FLUX upscale logic
    target_internal_size = 1024
    scale_factor = target_internal_size / max(orig_w, orig_h)
    if scale_factor > 1.0 or max(orig_w, orig_h) < 768: 
        new_w = int(orig_w * scale_factor)
        new_h = int(orig_h * scale_factor)
    else:
        new_w = orig_w
        new_h = orig_h
        
    opt_w = (new_w // 16) * 16
    opt_h = (new_h // 16) * 16
    
    if orig_w != opt_w or orig_h != opt_h:
        img_flux = img.resize((opt_w, opt_h), Image.LANCZOS)
    else:
        img_flux = img
    
    # Generate mask on the FLUX-sized image for precise inpainting
    img_cv = cv2.cvtColor(np.array(img_flux), cv2.COLOR_RGB2BGR)
    floor_mask = get_floor_mask(img_cv)
    inpaint_mask = generate_advanced_mask(img_cv.shape, floor_mask)
    
    # Calculate mask area percentage
    mask_area_percent = (np.sum(inpaint_mask > 0) / inpaint_mask.size) * 100
    
    inpaint_mask_pil = Image.fromarray(inpaint_mask)
    
    if seed is None:
        seed = random.randint(0, 2**32 - 1)
    generator = torch.Generator(device="cpu").manual_seed(seed)
    
    # Run FLUX pipeline
    result = ai_pipeline(
        prompt=prompt,
        image=img_flux,
        mask_image=inpaint_mask_pil,
        guidance_scale=30.0, 
        num_inference_steps=50, 
        max_sequence_length=256, # Reduced sequence length to save VRAM
        generator=generator,
    ).images[0]
    
    # Restore EXACT original dimensions for the final dataset export
    if orig_w != opt_w or orig_h != opt_h:
        result = result.resize((orig_w, orig_h), Image.LANCZOS)
        inpaint_mask_pil = inpaint_mask_pil.resize((orig_w, orig_h), Image.NEAREST)
        
    result_cv = cv2.cvtColor(np.array(result), cv2.COLOR_RGB2BGR)
    mask_cv = cv2.cvtColor(np.array(inpaint_mask_pil), cv2.COLOR_GRAY2BGR)
    
    return result_cv, mask_cv, prompt, seed, mask_area_percent



In [ ]:
#@title 15. Execute Pipeline
#@title 15. Execute Pipeline
metadata = []

# Double check output folder
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

if EXECUTION == 'preview':
    print("--- 🔍 PREVIEW MODE ---")
    print("Testing 5 different random images. No files will be saved.")
    
    if len(clean_images) > 0:
        sample_imgs = random.sample(clean_images, min(len(clean_images), 5))
        
        for idx, sample_img in enumerate(sample_imgs):
            print(f"\n[{idx+1}/5] Previewing augmentation on: {os.path.basename(sample_img)}")
            
            orig = cv2.imread(sample_img)
            if orig is None:
                print(f"  Warning: Could not read image {sample_img}. Skipping.")
                continue
                
            disp_orig = cv2.resize(orig, (OUTPUT_SIZE, OUTPUT_SIZE))
            
            if MODE == 'classical':
                result, _ = augment_classical(sample_img, difficulty=DIFFICULTY)
                if result is not None:
                    plot_images([disp_orig, cv2.resize(result, (OUTPUT_SIZE, OUTPUT_SIZE))], titles=['Original', 'Classical Overlay'])
                
            elif MODE == 'ai_inpainting':
                start_time = time.time()
                result, mask, prompt_used, seed_used, mask_area = augment_ai_inpainting(sample_img)
                exec_time = time.time() - start_time
                
                print(f"  Prompt: '{prompt_used}'")
                print(f"  Seed: {seed_used} | Mask Area: {mask_area:.2f}% | Time: {exec_time:.1f}s")
                
                disp_result = cv2.resize(result, (OUTPUT_SIZE, OUTPUT_SIZE))
                disp_mask = cv2.resize(mask, (OUTPUT_SIZE, OUTPUT_SIZE))
                plot_images([disp_orig, disp_mask, disp_result], titles=['Original', 'Mask', 'Generated (FLUX)'])
    else:
        print("No images found to preview.")

elif EXECUTION == 'batch':
    print("--- 🚀 BATCH MODE ---")
    print(f"Starting batch generation in {MODE} mode...")
    print(f"Generating {VARIANTS_PER_IMAGE} variants per image. Total expected: {len(clean_images) * VARIANTS_PER_IMAGE}")

    for img_path in tqdm(clean_images):
        orig_test = cv2.imread(img_path)
        if orig_test is None:
            print(f"\nWarning: Image {img_path} is unreadable. Skipping.")
            continue
            
        base_name = os.path.basename(img_path).split('.')[0]
        
        for i in range(VARIANTS_PER_IMAGE):
            out_name = f"{base_name}_dirty_{i+1}.jpg" # Match requested naming convention
            out_path = os.path.join(OUTPUT_FOLDER, out_name)
            
            try:
                start_time = time.time()
                if MODE == 'classical':
                    result, obj_used = augment_classical(img_path, num_objects=random.randint(1, MAX_OBJECTS), difficulty=DIFFICULTY)
                    prompt_used = ','.join(obj_used)
                    seed_used = "N/A"
                    mask_area = 0.0
                else:
                    # Retry logic for CUDA OOM or transient errors
                    try:
                        result, _, prompt_used, seed_used, mask_area = augment_ai_inpainting(img_path)
                    except Exception as e:
                        print(f"\nWarning: First attempt failed for {img_path}: {e}. Retrying once...")
                        time.sleep(2)
                        result, _, prompt_used, seed_used, mask_area = augment_ai_inpainting(img_path)
                    
                exec_time = time.time() - start_time
                
                if result is not None:
                    # Save at FULL original resolution
                    cv2.imwrite(out_path, result, [int(cv2.IMWRITE_JPEG_QUALITY), JPEG_QUALITY])
                    
                    print(f"\nGenerated: {out_name}")
                    print(f"  Prompt: {prompt_used}")
                    print(f"  Seed: {seed_used} | Mask Area: {mask_area:.2f}% | Time: {exec_time:.1f}s")
                    print(f"  Output Path: {out_path}")
                    
                    metadata.append({
                        'original_image': os.path.basename(img_path),
                        'generated_image': out_name,
                        'prompt': prompt_used,
                        'seed': seed_used,
                        'generation_time': round(exec_time, 1),
                        'mask_area_percent': round(mask_area, 2),
                        'model_name': 'FLUX.1-Fill-dev' if MODE == 'ai_inpainting' else 'Classical-CV-Overlay',
                        'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                    })
            except Exception as e:
                print(f"\nError processing {img_path} variant {i}: {e}. Skipping.")
                
    print("Batch generation completed!")
    
    if SAVE_METADATA and len(metadata) > 0:
        df = pd.DataFrame(metadata)
        csv_path = os.path.join(OUTPUT_FOLDER, 'generation_metadata.csv')
        df.to_csv(csv_path, index=False)
        print(f"Metadata saved to {csv_path}")


In [ ]:
#@title 16. Summary
print("--- GENERATION SUMMARY ---")
print(f"Execution run: {EXECUTION}")
if EXECUTION == 'batch':
    print(f"Mode used: {MODE}")
    print(f"Total synthetic dirty images generated: {len(metadata)}")
    print(f"Output folder: {OUTPUT_FOLDER}")
    print("Ready for MobileNetV2 training! 🚀")
else:
    print("Preview completed. Change EXECUTION to 'batch' to process all files.")
